In [ ]:
# This is the implementation of the radiative-transfer model
# calibrated in https://ieeexplore.ieee.org/document/11611206
# to simulate the emssivity of the Moon

In [ ]:
# Allows the modification of the files imported without having to restart kernel or re-import them
%load_ext autoreload
%autoreload 2
# to have interactive plots : pip install ipympl
%matplotlib ipympl

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
from astropy import units as u
from datetime import datetime
from astropy.time import Time
from astropy.coordinates import EarthLocation, AltAz, get_body
import pipeline_moon_functions as pmf

In [ ]:
nus_calib = np.array([89, 157, 183, 190])
Tb0 = np.array([261, 267, 268, 272])
Tb0_err = np.array([3, 6, 6, 3])
Tb1 = np.array([61, 81, 72, 83])
Tb1_err = np.array([3, 6, 6, 3])
phi0 = np.array([19.3, 11.2, 11.2, 10.1])
phi0_err = np.array([0.7, 0.8, 0.8, 0.4])
Tb2 = np.array([6, 9, 15, 11])
Tb2_err = np.array([1, 3, 3, 2])
# delta_Tb = np.array([4.1, 4.5, 4.6, 4.6])

def get_disc_integrated_Moon_TB_calib(i_nu, phi, dist_MS): # only calibrated Tb values
    # dist_MS in meters
    # print("Returning the disc-integrated bightness temperature for {} GHz.".format(nus_calib[i_nu]))
    Tb0_ = Tb0[i_nu]
    Tb1_ = Tb1[i_nu]
    phi0_ = phi0[i_nu]
    Tb2_ = Tb2[i_nu]
    delta_phi = np.radians(phi) - np.radians(phi0_)
    # delta_Tb_ = delta_Tb[i_nu]
    delta_Tb_ = (  Tb0_
                 + Tb1_ * (np.cos(delta_phi) - 1)
                 + Tb2_ * (np.cos(2 * delta_phi) + 1))
    Tb = (  delta_Tb_
          + delta_Tb_ * (pmf.AU_meters - dist_MS) / (2*pmf.AU_meters)) #(1.8 * 1e11)) # (2*pmf.AU_meters))
    
#     print(Tb0_)
#     print(Tb1_ * (np.cos(delta_phi) - 1))
#     print(Tb2_ * (np.cos(2 * delta_phi) + 1))
#     print(delta_Tb_ * (pmf.AU_meters - dist_MS) / (0.033 * 1e11))
    return Tb

def get_disc_integrated_Moon_TB_interp(nu, phi, dist_MS): # linear interpolation of calib values to get Tb for any nu in nu range
    # dist_MS in meters
    # print("Returning the disc-integrated bightness temperature for {} GHz.".format(nu))
    # the nu array has to contain sorted frequencies
    nu_GHz = nu/1e9
    nu_GHz = np.asarray(nu_GHz)
    if np.min(nu_GHz) < np.min(nus_calib) or np.max(nu_GHz) > np.max(nus_calib):
        raise ValueError("The given frequency nu = {} GHz is out of bounds.".format(nu_GHz))
    i_nus = np.zeros_like(nu_GHz, dtype=int)
    if np.isscalar(nu):
        while nus_calib[np.int_(i_nus)] < nu_GHz:
                i_nus += 1
    else:
        for i_, nu_ in enumerate(nu_GHz):
            while nus_calib[i_nus[i_]] < nu_:
                i_nus[i_:] += 1
    Tb_calib = []
    for i_ in range(len(nus_calib)):
        Tb_calib.append(get_disc_integrated_Moon_TB_calib(i_, phi, dist_MS))
    return np.interp(nu_GHz, nus_calib, Tb_calib)

In [ ]:
nu = 150e9 # Hz
phi = 10 # deg
dist_MS = pmf.AU_meters # m
Tb_test = get_disc_integrated_Moon_TB_interp(nu, phi, dist_MS)

In [ ]:
Tb_test

In [ ]:
ObsSite = pmf.LaPuna_QUBIC
Obs_Site = EarthLocation(lat=ObsSite["lat"], lon=ObsSite["lon"], height=ObsSite["height"])
longitude = float(Obs_Site.lon/u.deg)
latitude = float(Obs_Site.lat/u.deg)
# times at which we want to know the position of the Moon
tinit = 1785020764 # random test with 25/07/2026 23:06:04
tt = np.linspace(0, 1000, 1001)
date_obs = str(datetime.utcfromtimestamp(tinit))
print('Observations started at: {} UTC (old)'.format(date_obs))

alltimes = Time(date_obs) + tt*u.second

### Moon
moon_observed = get_body('Moon', alltimes, Obs_Site)

# Computing the mean angular size of the Moon
moon_dist = moon_observed.distance.value * pmf.AU_meters # m
moon_radius = 1737.4 * 1e3 # m
moon_angsize = np.mean(np.arctan(2 * moon_radius / moon_dist)) # radians
moon_solid_angle = np.pi * moon_radius**2 / moon_dist**2 # steradian

In [ ]:
h = 6.62607015e-34 # J.Hz-1
c = 299792458 # m.s-1
k = 1.380649e-23 # J.K-1
def Tb2I(Tb, nu): # per unit surface area per unit time per unit solid angle and in the frequency range between ν and ν + dν
    # W m-2 Hz-1 sr-1
    return 2 * h * nu**3 / c**2 / (np.exp(h * nu / (k * Tb)) - 1)
    # Rayleigh-Jeans approx
    # return 2 * k * Tb * nu**2 / c**2
def nu2P_pix(nu_interval, nside, phi, dist_MS): # power emitted by Moon in a frequency range nu_interval per pixel
    n_nus = 100
    nus_bins = np.linspace(nu_interval[0], nu_interval[1], n_nus + 1)
    nus = (nus_bins[1:] + nus_bins[:-1])/2
    Tbs = get_disc_integrated_Moon_TB_interp(nus, phi, dist_MS)
    I_nus = Tb2I(Tbs, nus)
    # we get the solid angle of pixels
    pixel_solid_angle = hp.nside2pixarea(nside) # radians**2 = steradians (checked with (180/pi)**2 conversion factor)
    print(pixel_solid_angle)
    # we integrate the emission for each nu and in the solid angle of a pixel
    P = np.sum(I_nus * (nus_bins[1:] - nus_bins[:-1])) * pixel_solid_angle
    return P
def get_moon_map(nu_interval, nside, moon_pos, moon_angsize, phi, dist_MS):
    power_pixel = nu2P_pix(nu_interval, nside, phi, dist_MS)
    print("each pixel at nside = {} precision holds a power P = {} W for 1 m2 of detecting area".format(nside, power_pixel))
    # in order to have a good Moon map, we need to account for pixels partially covering the Moon
    # to do so we can use a large NSIDE, put the pixels to one when in the Moon radius
    # and then degrade the resolution!
    high_nside = 16 * nside
    moon_pix = hp.query_disc(nside=high_nside, vec=moon_pos, radius=moon_angsize)
    print(len(moon_pix))
    map_moon = np.zeros(hp.nside2npix(high_nside))
    map_moon[moon_pix] = 1 # it will later be multiplied by power_pix
    map_moon = hp.ud_grade(map_in=map_moon, nside_out=nside) # faster
    # map_moon = hp.sphtfunc.harmonic_ud_grade(map_in=map_moon, nside_out=nside) # not available in healpy 1.19.0
    # map_moon *= power_pixel
    return map_moon # this should account for the coverage of the Moon

In [ ]:
print(0.984 * pmf.AU_meters, 1.016 * pmf.AU_meters)
print(0.033 * 1e11)
print(pmf.AU_meters/(0.033 * 1e11))

In [ ]:
phi = np.linspace(-100, 50, 1000)
dist_MS = np.array([0.984, 1, 1.016]) * pmf.AU_meters
ls = ["--", "-", "-."]
colors = ["red", "green", "blue"]
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
for i_freq in range(len(nus_calib) - 1):
    Tbs_dist = []
    for i_dist in range(len(dist_MS)):
        Tbs = get_disc_integrated_Moon_TB_calib(i_freq, phi, dist_MS[i_dist])
        Tbs_dist.append(Tbs)
        axs[0].plot(phi, Tbs, c=colors[i_freq], ls=ls[i_dist], label="{} GHz".format(nus_calib[i_freq]))
    axs[1].plot(phi, Tbs_dist[0] - Tbs_dist[1], c=colors[i_freq], label="{} GHz".format(nus_calib[i_freq]))
plt.legend()
plt.show()

In [ ]:
phi = 0
dist_MS = np.linspace(0.984, 1.016, 10) * pmf.AU_meters
colors = ["red", "green", "blue"]
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.set_title("{} GHz".format(nus_calib[i_freq]))
Tbs_dist = []
i_freq = 0
ref = get_disc_integrated_Moon_TB_calib(i_freq, phi, 1*pmf.AU_meters)
for i_dist in range(len(dist_MS)):
    Tbs = get_disc_integrated_Moon_TB_calib(i_freq, phi, dist_MS[i_dist])
    Tbs_dist.append(Tbs)
Tbs_dist = np.array(Tbs_dist)
ax.plot(dist_MS, Tbs_dist - ref, c=colors[i_freq])
plt.legend()
plt.show()

In [ ]:
phis = np.linspace(-90, 90, 5)
dist_MS = 1 * pmf.AU_meters
colors = ["red", "green", "blue"]
fig1, ax1 = plt.subplots(1, 1, figsize=(10, 5))
fig2, ax2 = plt.subplots(1, 1, figsize=(10, 5))
i_freqs = np.arange(4)
for i_phi, phi in enumerate(phis):
    Tbs_freq = []
    I_nu = []
    for i_freq in i_freqs:
        Tbs = get_disc_integrated_Moon_TB_calib(i_freq, phi, dist_MS)
        Tbs_freq.append(Tbs)
        I_nu.append(Tb2I(Tbs, nus_calib[i_freq]))
    # Tbs_freq = np.array(Tbs_freq)
    ax1.plot(nus_calib, Tbs_freq, label="Lunar phase angle = {}°".format(phi))
    ax2.plot(nus_calib, I_nu, label="Lunar phase angle = {}°".format(phi))
ax1.set_title("Tb [K] vs nu [GHz]")
ax2.set_title("I_nu [W m-2 Hz-1 sr-1] vs nu [GHz]")
fig1.legend()
fig2.legend()
plt.show()

In [ ]:
phis = np.linspace(-90, 90, 5)
dist_MS = 1 * pmf.AU_meters
colors = ["red", "green", "blue"]
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.set_title("{} GHz".format(nus_calib[i_freq]))
nus = np.linspace(np.min(nus_calib) + 1, np.max(nus_calib) - 1, 20) * 1e9 # Hz
# i_freqs = np.arange(4)
for i_phi, phi in enumerate(phis):
    Tbs_freq = get_disc_integrated_Moon_TB_interp(nus, phi, dist_MS)
    ax.plot(nus, Tbs_freq, label="Lunar phase angle = {}°".format(phi))
plt.legend()
plt.show()

In [ ]:
nside = 512
az_moon = np.float_(90) # deg
el_moon = np.float_(50) # deg
moon_pos = pmf.spherical2cartesian(rho=1, theta=az_moon, phi=el_moon, coord="horizontal")
Tb = 250 # K
nu_interval = np.array([130, 170]) * 1e9 # Hz
map_moon = get_moon_map(nu_interval, nside, moon_pos, moon_angsize, phi, dist_MS)

In [ ]:
hp.mollview(map_moon, xsize=1000, cmap="grey") # xsize=10000 with nside = 1024 to really see a circle
plt.show()

In [ ]:
hp.gnomview(map_moon, rot=[az_moon, el_moon], reso=0.1, xsize=1000, cmap="grey")
plt.show()